In [8]:
import time
import chess
import chess.pgn
import chess.svg
import ipywidgets as widgets
from IPython.display import display, clear_output, SVG

# ==========================================
# 1. MATERIAL & POSITIONAL TABLES
# ==========================================
PIECE_VALUES = {
    chess.PAWN: 100, chess.KNIGHT: 320, chess.BISHOP: 330,
    chess.ROOK: 500, chess.QUEEN: 900, chess.KING: 20000
}

PAWN_PST = [
     0,  0,  0,  0,  0,  0,  0,  0,
     5, 10, 10,-20,-20, 10, 10,  5,
     5, -5,-10,  0,  0,-10, -5,  5,
    10, 10, 15, 25, 25, 15, 10, 10,
    20, 20, 30, 40, 40, 30, 20, 20,
    40, 40, 50, 60, 60, 50, 40, 40,
    80, 80, 90,100,100, 90, 80, 80,
     0,  0,  0,  0,  0,  0,  0,  0
]
KNIGHT_PST = [
   -50,-40,-30,-30,-30,-30,-40,-50,
   -40,-20,  0,  5,  5,  0,-20,-40,
   -30,  5, 10, 15, 15, 10,  5,-30,
   -30,  0, 15, 20, 20, 15,  0,-30,
   -30,  5, 15, 20, 20, 15,  5,-30,
   -30,  0, 10, 15, 15, 10,  0,-30,
   -40,-20,  0,  0,  0,  0,-20,-40,
   -50,-40,-30,-30,-30,-30,-40,-50
]
BISHOP_PST = [
   -20,-10,-10,-10,-10,-10,-10,-20,
   -10,  5,  0,  0,  0,  0,  5,-10,
   -10, 10, 10, 10, 10, 10, 10,-10,
   -10,  0, 10, 10, 10, 10,  0,-10,
   -10,  5,  5, 10, 10,  5,  5,-10,
   -10,  0,  5, 10, 10,  5,  0,-10,
   -10,  0,  0,  0,  0,  0,  0,-10,
   -20,-10,-10,-10,-10,-10,-10,-20
]
ROOK_PST = [
     0,  0,  0,  5,  5,  0,  0,  0,
    -5,  0,  0,  0,  0,  0,  0, -5,
    -5,  0,  0,  0,  0,  0,  0, -5,
    -5,  0,  0,  0,  0,  0,  0, -5,
    -5,  0,  0,  0,  0,  0,  0, -5,
    -5,  0,  0,  0,  0,  0,  0, -5,
     5, 10, 10, 10, 10, 10, 10,  5,
     0,  0,  0,  0,  0,  0,  0,  0
]
QUEEN_PST = [
   -20,-10,-10, -5, -5,-10,-10,-20,
   -10,  0,  5,  0,  0,  0,  0,-10,
   -10,  5,  5,  5,  5,  5,  0,-10,
    -5,  0,  5,  5,  5,  5,  0, -5,
     0,  0,  5,  5,  5,  5,  0, -5,
   -10,  0,  5,  5,  5,  5,  0,-10,
   -10,  0,  0,  0,  0,  0,  0,-10,
   -20,-10,-10, -5, -5,-10,-10,-20
]
KING_MG_PST = [
    20, 30, 10,  0,  0, 10, 30, 20,
    20, 20,  0,  0,  0,  0, 20, 20,
   -10,-20,-20,-20,-20,-20,-20,-10,
   -20,-30,-30,-40,-40,-30,-30,-20,
   -30,-40,-40,-50,-50,-40,-40,-30,
   -30,-40,-40,-50,-50,-40,-40,-30,
   -30,-40,-40,-50,-50,-40,-40,-30,
   -30,-40,-40,-50,-50,-40,-40,-30
]
KING_EG_PST = [
   -50,-40,-30,-20,-20,-30,-40,-50,
   -30,-20,-10,  0,  0,-10,-20,-30,
   -30,-10, 20, 30, 30, 20,-10,-30,
   -30,-10, 30, 40, 40, 30,-10,-30,
   -30,-10, 30, 40, 40, 30,-10,-30,
   -30,-10, 20, 30, 30, 20,-10,-30,
   -30,-30,  0,  0,  0,  0,-30,-30,
   -50,-30,-30,-30,-30,-30,-30,-50
]

PST_TABLES = {
    chess.PAWN: PAWN_PST, chess.KNIGHT: KNIGHT_PST,
    chess.BISHOP: BISHOP_PST, chess.ROOK: ROOK_PST,
    chess.QUEEN: QUEEN_PST,
}

TT = {}
EXACT, LOWERBOUND, UPPERBOUND = 0, 1, 2

# ==========================================
# 2. OPTIMIZED ENGINE ENGINE SEARCH
# ==========================================
def evaluate_board(board: chess.Board) -> int:
    if board.is_stalemate() or board.is_insufficient_material():
        return 0

    score = 0
    phase = 0

    for square, piece in board.piece_map().items():
        pt = piece.piece_type
        color = piece.color
        
        if pt in (chess.KNIGHT, chess.BISHOP): phase += 1
        elif pt == chess.ROOK: phase += 2
        elif pt == chess.QUEEN: phase += 4

        base_val = PIECE_VALUES[pt]
        sq_idx = square if color == chess.WHITE else chess.square_mirror(square)

        if pt == chess.KING:
            pst_val = int(((KING_MG_PST[sq_idx] * phase) + (KING_EG_PST[sq_idx] * (24 - phase))) / 24)
        else:
            pst_val = PST_TABLES[pt][sq_idx]

        if color == chess.WHITE:
            score += base_val + pst_val
        else:
            score -= base_val + pst_val

    return score

def order_moves(board, moves, hash_move=None):
    def score_move(move):
        if move == hash_move:
            return 10000000
            
        score = 0
        if board.is_capture(move):
            if board.is_en_passant(move):
                return 100005
            
            victim = board.piece_at(move.to_square)
            attacker = board.piece_at(move.from_square)
            if victim and attacker:
                score = 100000 + PIECE_VALUES[victim.piece_type] - PIECE_VALUES[attacker.piece_type]
        
        if move.promotion:
            score += 90000 + PIECE_VALUES[move.promotion]
            
        return score

    return sorted(moves, key=score_move, reverse=True)

def quiescence_search(board, alpha, beta, color):
    stand_pat = evaluate_board(board) * color
    if stand_pat >= beta:
        return beta
    if alpha < stand_pat:
        alpha = stand_pat

    captures = list(board.generate_legal_captures())
    for move in order_moves(board, captures):
        board.push(move)
        score = -quiescence_search(board, -beta, -alpha, -color)
        board.pop()

        if score >= beta:
            return beta
        if score > alpha:
            alpha = score

    return alpha

def negamax(board, depth, alpha, beta, color):
    original_alpha = alpha
    hash_key = board.fen()
    hash_move = None

    if hash_key in TT:
        tt_depth, tt_flag, tt_score, tt_move = TT[hash_key]
        if tt_depth >= depth:
            if tt_flag == EXACT: return tt_score
            if tt_flag == LOWERBOUND and tt_score >= beta: return tt_score
            if tt_flag == UPPERBOUND and tt_score <= alpha: return tt_score
        hash_move = tt_move

    if depth == 0:
        return quiescence_search(board, alpha, beta, color)

    moves = list(board.legal_moves)
    if not moves:
        if board.is_check():
            return -99999 + depth
        return 0

    best_move = None
    best_score = -9999999
    ordered_moves = order_moves(board, moves, hash_move)

    for move in ordered_moves:
        board.push(move)
        score = -negamax(board, depth - 1, -beta, -alpha, -color)
        board.pop()

        if score > best_score:
            best_score = score
            best_move = move

        alpha = max(alpha, best_score)
        if alpha >= beta:
            break

    if best_score <= original_alpha:
        flag = UPPERBOUND
    elif best_score >= beta:
        flag = LOWERBOUND
    else:
        flag = EXACT
    TT[hash_key] = (depth, flag, best_score, best_move)

    return best_score

def get_ai_move(board: chess.Board, max_depth: int = 6) -> chess.Move:
    best_move = None
    color = 1 if board.turn == chess.WHITE else -1

    for d in range(1, max_depth + 1):
        score = negamax(board, d, -9999999, 9999999, color)
        tt_entry = TT.get(board.fen())
        if tt_entry:
            best_move = tt_entry[3]
        if abs(score) > 90000:  # Mate sequence found, don't waste time searching deeper
            break
            
    return best_move

# ==========================================
# 3. FUNCTION 1: SIMULATE GAME & EXPORT PGN
# ==========================================
def simulate_and_export_pgn(filename="game.pgn", white_depth=5, black_depth=5, event_name="AI Match"):
    """
    Simulates a full game between two depths in background and outputs a PGN file.
    """
    board = chess.Board()
    game = chess.pgn.Game()
    
    # Metadata
    game.headers["Event"] = event_name
    game.headers["White"] = f"Python Engine (Depth {white_depth})"
    game.headers["Black"] = f"Python Engine (Depth {black_depth})"
    
    node = game
    TT.clear()
    move_count = 1

    print(f"Baking full game... (White: Depth {white_depth} | Black: Depth {black_depth})")
    start_total_time = time.time()

    while not board.is_game_over(claim_draw=True):
        turn_depth = white_depth if board.turn == chess.WHITE else black_depth
        
        start_move = time.time()
        ai_move = get_ai_move(board, max_depth=turn_depth)
        elapsed = time.time() - start_move

        if not ai_move:
            break

        board.push(ai_move)
        node = node.add_main_variation(ai_move)
        
        current_turn = "White" if board.turn == chess.BLACK else "Black"
        print(f"Move {move_count}: {current_turn} played {ai_move.uci()} ({elapsed:.2f}s)")
        
        if board.turn == chess.WHITE:
            move_count += 1

    game.headers["Result"] = board.result(claim_draw=True)
    
    # Save file
    with open(filename, "w") as f:
        exporter = chess.pgn.FileExporter(f)
        game.accept(exporter)

    total_time = time.time() - start_total_time
    print(f"\nCompleted in {total_time:.2f}s! Game saved to '{filename}' with Result: {board.result(claim_draw=True)}")
    return filename

# ==========================================
# 4. FUNCTION 2: INTERACTIVE PGN VIEWER
# ==========================================
class PGNViewer:
    """
    Reads a PGN file and provides an interactive UI with Back / Next buttons.
    """
    def __init__(self, pgn_filename):
        with open(pgn_filename) as f:
            self.game = chess.pgn.read_game(f)

        self.history = []
        self.moves = []
        
        # Parse all moves and build positions
        temp_board = self.game.board()
        self.history.append(temp_board.copy())
        
        for move in self.game.mainline_moves():
            temp_board.push(move)
            self.history.append(temp_board.copy())
            self.moves.append(move)

        self.current_idx = 0

        # UI Elements
        self.output_area = widgets.Output()
        self.btn_first = widgets.Button(description="|< First", button_style='info')
        self.btn_prev = widgets.Button(description="< Prev", button_style='primary')
        self.btn_next = widgets.Button(description="Next >", button_style='primary')
        self.btn_last = widgets.Button(description="Last >|", button_style='info')

        self.btn_first.on_click(self.go_first)
        self.btn_prev.on_click(self.go_prev)
        self.btn_next.on_click(self.go_next)
        self.btn_last.on_click(self.go_last)

        self.controls = widgets.HBox([self.btn_first, self.btn_prev, self.btn_next, self.btn_last])
        
        display(self.controls)
        display(self.output_area)
        self.render()

    def render(self):
        with self.output_area:
            clear_output(wait=True)
            board = self.history[self.current_idx]
            last_m = self.moves[self.current_idx - 1] if self.current_idx > 0 else None
            
            svg = chess.svg.board(
                board, 
                size=400, 
                lastmove=last_m,
                check=board.king(board.turn) if board.is_check() else None
            )
            display(SVG(svg))
            
            # Text Status
            if self.current_idx == 0:
                print("Start Position")
            else:
                last_played = self.moves[self.current_idx - 1]
                move_num = (self.current_idx + 1) // 2
                color = "White" if self.current_idx % 2 != 0 else "Black"
                print(f"Move {self.current_idx}/{len(self.moves)}: {move_num}. {color} played {last_played.uci()}")
            
            if self.current_idx == len(self.history) - 1:
                print(f"Game Finished! Result: {self.game.headers.get('Result', '*')}")

    def go_first(self, b):
        self.current_idx = 0
        self.render()

    def go_prev(self, b):
        if self.current_idx > 0:
            self.current_idx -= 1
            self.render()

    def go_next(self, b):
        if self.current_idx < len(self.history) - 1:
            self.current_idx += 1
            self.render()

    def go_last(self, b):
        self.current_idx = len(self.history) - 1
        self.render()

def load_and_view_pgn(filename="game.pgn"):
    return PGNViewer(filename)

In [9]:
simulate_and_export_pgn(filename="AI_MMQsPst_d6d4.pgn", white_depth=6, black_depth=4)


Baking full game... (White: Depth 6 | Black: Depth 4)
Move 1: White played b1c3 (8.98s)
Move 1: Black played b8c6 (0.00s)
Move 2: White played e2e4 (26.57s)
Move 2: Black played e7e5 (0.00s)
Move 3: White played g1f3 (29.47s)
Move 3: Black played g8f6 (0.00s)
Move 4: White played d2d4 (42.62s)
Move 4: Black played e5d4 (0.00s)
Move 5: White played f3d4 (17.10s)
Move 5: Black played f8b4 (0.00s)
Move 6: White played d4c6 (8.29s)
Move 6: Black played b4c3 (0.00s)
Move 7: White played b2c3 (1.77s)
Move 7: Black played d7c6 (0.00s)
Move 8: White played d1d8 (18.41s)
Move 8: Black played e8d8 (0.00s)
Move 9: White played c1g5 (7.03s)
Move 9: Black played h8e8 (0.00s)
Move 10: White played f1d3 (11.87s)
Move 10: Black played d8d7 (0.00s)
Move 11: White played e1c1 (19.39s)
Move 11: Black played d7d6 (0.00s)
Move 12: White played f2f4 (46.88s)
Move 12: Black played h7h6 (0.00s)
Move 13: White played e4e5 (6.30s)
Move 13: Black played d6d5 (0.00s)
Move 14: White played e5f6 (10.77s)
Move 14: B

'AI_MMQsPst_d6d4.pgn'

In [10]:
load_and_view_pgn("AI_MMQsPst_d6d4.pgn")

Output()